# Agente IA Triage de Síntomas — Clase 90 min

**Enfoque:** Musk (simple, rápido) + FinOps (gpt-4o-mini) + Zero-Hallucination (structured output).

**NO hace diagnóstico.** Solo clasifica en ROJO / AMARILLO / VERDE y recomienda acción.

## 1. Setup

In [ ]:
import os
import json
import time
from enum import Enum
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from openai import OpenAI

load_dotenv()

USE_LOCAL = os.getenv('USE_LOCAL', 'False').lower() == 'true'
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
OLLAMA_HOST = os.getenv('OLLAMA_HOST', 'http://localhost:11434')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'llama3.2:3b')

print(f'Modo: {"LOCAL (Ollama)" if USE_LOCAL else f"OpenAI ({OPENAI_MODEL})"}')

## 2. Config FinOps

Precios OpenAI (por 1M tokens):
- **gpt-4o-mini**: input $0.15 / output $0.60 ← **nuestro default**
- gpt-4o: input $2.50 / output $10.00 (16x más caro)

In [ ]:
PRICE_INPUT_PER_1M = 0.15   # gpt-4o-mini
PRICE_OUTPUT_PER_1M = 0.60

MAX_TOKENS = 300
TEMPERATURE = 0.1
TIMEOUT = 10

CONFIDENCE_THRESHOLD = 0.85  # debajo de esto, escalar a humano

def calcular_costo(input_tokens: int, output_tokens: int) -> float:
    return (input_tokens * PRICE_INPUT_PER_1M + output_tokens * PRICE_OUTPUT_PER_1M) / 1_000_000

## 3. Schema (Zero-Hallucination via Pydantic)

El LLM **NO puede** inventar formato: está obligado a retornar este JSON.

In [ ]:
class NivelTriage(str, Enum):
    ROJO = 'ROJO'       # Emergencia: ir YA / llamar ambulancia
    AMARILLO = 'AMARILLO'  # Urgente: atención hoy
    VERDE = 'VERDE'     # No urgente: casa o teleconsulta

class AccionRecomendada(str, Enum):
    LLAMAR_AMBULANCIA = 'LLAMAR_AMBULANCIA'
    IR_EMERGENCIA = 'IR_EMERGENCIA'
    CONSULTA_HOY = 'CONSULTA_HOY'
    TELECONSULTA = 'TELECONSULTA'
    AUTOCUIDADO = 'AUTOCUIDADO'

class TriageResult(BaseModel):
    nivel: NivelTriage
    accion: AccionRecomendada
    razon: str = Field(description='Justificación breve, max 2 líneas')
    senales_alarma: list[str] = Field(default_factory=list, description='Síntomas que motivaron el nivel')
    confianza: float = Field(ge=0.0, le=1.0, description='0.0-1.0')
    disclaimer: str = 'Este resultado NO es un diagnóstico médico. Consulte a un profesional de salud.'

## 4. System Prompt (Manchester Triage simplificado)

In [ ]:
SYSTEM_PROMPT = '''Eres un agente de triage médico basado en el protocolo Manchester Triage System.
Tu ÚNICA función es CLASIFICAR síntomas en niveles de urgencia. NO diagnosticas, NO recetas.

NIVELES:
- ROJO: riesgo vital inmediato. Ejemplos: dolor torácico con irradiación, signos de ACV (debilidad/dificultad hablar),
  sangrado abundante, dificultad respiratoria severa, pérdida de consciencia, trauma grave, embarazo con sangrado.
- AMARILLO: requiere atención el mismo día, no puede esperar. Ejemplos: fiebre alta persistente,
  corte profundo con sangrado controlado, dolor moderado-severo, infección con empeoramiento.
- VERDE: puede manejarse en casa o teleconsulta. Ejemplos: resfriado, dolor leve, malestar general sin signos de alarma.

REGLAS ESTRICTAS:
1. Ante duda entre dos niveles, ELIGE el MÁS URGENTE (principio de precaución).
2. Si los síntomas son ambiguos o insuficientes, confianza <= 0.7.
3. NUNCA diagnostiques enfermedad específica en el campo razon.
4. NUNCA recetes medicamentos ni dosis.
5. Responde SOLO con el JSON del schema. Nada de texto extra.

ACCIÓN por nivel:
- ROJO + señales críticas → LLAMAR_AMBULANCIA
- ROJO moderado → IR_EMERGENCIA
- AMARILLO → CONSULTA_HOY
- VERDE + síntomas claros → AUTOCUIDADO
- VERDE + consulta recomendable → TELECONSULTA'''

FEW_SHOT_EXAMPLES = [
    {
        'input': 'Hombre 60 años, dolor opresivo en pecho con sudoración, irradia al brazo izquierdo, 15 min',
        'output': {
            'nivel': 'ROJO', 'accion': 'LLAMAR_AMBULANCIA',
            'razon': 'Síntomas compatibles con evento cardiovascular agudo. Requiere atención inmediata.',
            'senales_alarma': ['dolor torácico irradiado', 'sudoración', 'edad mayor'],
            'confianza': 0.95
        }
    },
    {
        'input': 'Mujer 25 años, estornudos y congestión desde hace 2 días, sin fiebre',
        'output': {
            'nivel': 'VERDE', 'accion': 'AUTOCUIDADO',
            'razon': 'Cuadro catarral leve sin signos de alarma. Manejo en casa.',
            'senales_alarma': [],
            'confianza': 0.92
        }
    }
]

## 5. Cliente OpenAI + función principal

In [ ]:
client = OpenAI() if not USE_LOCAL else None

def build_messages(sintomas: str) -> list[dict]:
    examples_text = '\n\n'.join(
        f'EJEMPLO:\nSíntomas: {ex["input"]}\nRespuesta: {json.dumps(ex["output"], ensure_ascii=False)}'
        for ex in FEW_SHOT_EXAMPLES
    )
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT + '\n\n' + examples_text},
        {'role': 'user', 'content': f'Síntomas del paciente: {sintomas}\n\nResponde SOLO con el JSON del schema.'}
    ]

def triage_openai(sintomas: str) -> tuple[TriageResult, dict]:
    t0 = time.time()
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=build_messages(sintomas),
        response_format={'type': 'json_object'},
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=TIMEOUT,
    )
    latencia = time.time() - t0
    raw = response.choices[0].message.content
    data = json.loads(raw)
    result = TriageResult(**data)
    meta = {
        'latencia_s': round(latencia, 2),
        'input_tokens': response.usage.prompt_tokens,
        'output_tokens': response.usage.completion_tokens,
        'costo_usd': calcular_costo(response.usage.prompt_tokens, response.usage.completion_tokens),
        'modelo': OPENAI_MODEL,
    }
    return result, meta

## 6. Fallback Local (Ollama) — $0 por triage

In [ ]:
import requests

def triage_ollama(sintomas: str) -> tuple[TriageResult, dict]:
    t0 = time.time()
    payload = {
        'model': OLLAMA_MODEL,
        'messages': build_messages(sintomas),
        'format': 'json',
        'options': {'temperature': TEMPERATURE, 'num_predict': MAX_TOKENS},
        'stream': False,
    }
    r = requests.post(f'{OLLAMA_HOST}/api/chat', json=payload, timeout=60)
    r.raise_for_status()
    latencia = time.time() - t0
    data = json.loads(r.json()['message']['content'])
    result = TriageResult(**data)
    meta = {
        'latencia_s': round(latencia, 2),
        'input_tokens': r.json().get('prompt_eval_count', 0),
        'output_tokens': r.json().get('eval_count', 0),
        'costo_usd': 0.0,
        'modelo': OLLAMA_MODEL,
    }
    return result, meta

## 7. Guardrails (validación + escalamiento humano)

In [ ]:
def triage(sintomas: str) -> dict:
    if not sintomas or len(sintomas.strip()) < 10:
        return {'error': 'Descripción insuficiente. Proporcione más detalles.'}

    fn = triage_ollama if USE_LOCAL else triage_openai
    try:
        result, meta = fn(sintomas)
    except Exception as e:
        return {'error': f'Fallo del agente: {e}', 'escalar_humano': True}

    salida = {**result.model_dump(), **meta}

    # Escalamiento automático si confianza baja
    if result.confianza < CONFIDENCE_THRESHOLD:
        salida['escalar_humano'] = True
        salida['motivo_escalamiento'] = f'Confianza {result.confianza:.2f} < {CONFIDENCE_THRESHOLD}'
    else:
        salida['escalar_humano'] = False

    return salida

## 8. Demo en Vivo

In [ ]:
casos_path = Path('..') / 'data' / 'casos_prueba.json'
casos = json.loads(casos_path.read_text(encoding='utf-8'))

print(f'Total casos: {len(casos)}\n')
for c in casos:
    print(f"[{c['id']}] esperado={c['esperado']}: {c['descripcion'][:80]}...")

In [ ]:
from tabulate import tabulate

resultados = []
costo_total = 0.0
latencia_total = 0.0
aciertos = 0

for caso in casos:
    r = triage(caso['descripcion'])
    if 'error' in r:
        print(f"[{caso['id']}] ERROR: {r['error']}")
        continue
    acierto = r['nivel'] == caso['esperado']
    aciertos += int(acierto)
    costo_total += r['costo_usd']
    latencia_total += r['latencia_s']
    resultados.append({
        'id': caso['id'],
        'esperado': caso['esperado'],
        'predicho': r['nivel'],
        'accion': r['accion'],
        'conf': f"{r['confianza']:.2f}",
        'escalar': 'SI' if r['escalar_humano'] else 'no',
        'tokens': f"{r['input_tokens']}+{r['output_tokens']}",
        'lat(s)': r['latencia_s'],
        'costo($)': f"{r['costo_usd']:.6f}",
        'OK': '✓' if acierto else '✗',
    })

print(tabulate(resultados, headers='keys', tablefmt='simple'))

## 9. Métricas FinOps

In [ ]:
n = len(resultados)
print(f"\n=== MÉTRICAS ({n} casos) ===")
print(f"Accuracy:         {aciertos}/{n} = {aciertos/n*100:.1f}%")
print(f"Costo total:      ${costo_total:.6f}")
print(f"Costo promedio:   ${costo_total/n:.6f}/caso")
print(f"Latencia prom.:   {latencia_total/n:.2f}s")

print(f"\n=== PROYECCIÓN ===")
for pacientes in [100, 1000, 10000, 100000]:
    print(f"{pacientes:>6} pacientes/día = ${costo_total/n * pacientes:.2f}/día")

print(f"\n=== VS ENFERMERA ===")
costo_enfermera_hora_usd = 13  # ~S/50/hora
triages_por_hora = 10
costo_triage_humano = costo_enfermera_hora_usd / triages_por_hora
costo_triage_ia = costo_total / n if n else 0
if costo_triage_ia > 0:
    print(f"Humano: ${costo_triage_humano:.2f}/triage")
    print(f"IA:     ${costo_triage_ia:.6f}/triage")
    print(f"Ratio:  {costo_triage_humano/costo_triage_ia:.0f}x más barato")

## 10. Demo interactiva — prueba tu propio caso

In [ ]:
mi_caso = 'Mujer 45 años, dolor de cabeza intenso hace 1 hora, visión borrosa, mareo'

resultado = triage(mi_caso)
print(json.dumps(resultado, indent=2, ensure_ascii=False))

## 11. Cierre — Principios Musk aplicados

| Principio | Cómo lo aplicamos |
|-----------|---------------------|
| **Cuestionar** | ¿Necesitamos RAG? NO. ¿Fine-tuning? NO. ¿Frontend? NO. |
| **Eliminar** | 1 archivo, 1 modelo, 1 prompt. Cero frameworks innecesarios. |
| **Simplificar** | 3 niveles, 5 acciones. Ni uno más. |
| **Acelerar** | 90 min clase → agente funcionando + métricas FinOps |
| **Automatizar** | Solo después de validar manualmente (escalamiento humano). |

### Próximos pasos (si fuera producto real)
1. Validar con 100 casos reales supervisados por médico
2. A/B test vs triage humano
3. Integrar WhatsApp Business API
4. Dashboard multi-tenant para clínicas
5. Certificación regulatoria (DIGEMID/MINSA)